In [1]:
# This notebook applies the following basic Machine Learning models:
# Logistic Regression, SVM, KNN and Decision Trees
###

# 0. Preparation
###
# Importing libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from sklearn.metrics import classification_report, f1_score
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split
from sklearn import svm, neighbors

In [2]:
# Mounting GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Reading data file from GoogleDrive
df = pd.read_pickle("/content/drive/My Drive/Data Science/Team Project X-Rays/Dataframes/df_basic_1.3_no_NA.pkl")
df.head()

# Define data name
df_name = "Baseline 1.3_no_NA"

# Define random subsample for computation efficiency
#df = df.sample(500)


In [4]:
# Explore Data
###

# Check data type is DataFrame
print(type(df))

# Show dimensions
print(df.shape)

# Show labels
print(df.Case.value_counts())

# Check distributions after normalisation
df.describe()

# Why do we only have 1530 Columns: After Masking, columns with more than 80% NA values were deleted
#  and the remaining NA values were replaced by the mean value

<class 'pandas.core.frame.DataFrame'>
(21105, 1530)
Case
Normal             10191
Lung_Opacity        6012
COVID               3564
Viral Pneumonia     1338
Name: count, dtype: int64


,PX_407,PX_408,PX_409,PX_410,PX_411,PX_423,PX_424,PX_425,PX_426,PX_427,...,PX_3257,PX_3274,PX_3275,PX_3316,PX_3317,PX_3318,PX_3319,PX_3320,PX_3382,PX_3383
count,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,...,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000,21105.000000
mean,0.435316,0.430157,0.428870,0.431693,0.438176,0.422108,0.416397,0.412695,0.416362,0.422797,...,0.463969,0.437964,0.438279,0.429313,0.430043,0.432121,0.439167,0.443240,0.434326,0.439704
std,0.071123,0.073707,0.073789,0.071801,0.067969,0.066873,0.070694,0.070965,0.070351,0.067652,...,0.085615,0.083233,0.082796,0.085613,0.090241,0.092166,0.092109,0.087637,0.083251,0.082836
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.015686,0.023529,0.019608,0.019608,0.035294,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003922,0.003922,0.003922,0.003922
25%,0.435316,0.430157,0.428870,0.431693,0.438176,0.422108,0.416397,0.412695,0.416362,0.422797,...,0.463969,0.437964,0.438279,0.429313,0.430043,0.432121,0.439167,0.443240,0.434326,0.439705
50%,0.435316,0.430157,0.428870,0.431693,0.438176,0.422108,0.416397,0.412695,0.416362,0.422797,...,0.463969,0.437964,0.438279,0.429313,0.430043,0.432121,0.439167,0.443240,0.434326,0.439705
75%,0.435316,0.430157,0.428870,0.431693,0.438176,0.422108,0.416397,0.412695,0.416362,0.422797,...,0.463969,0.437964,0.438279,0.429313,0.430043,0.432121,0.439167,0.443240,0.434326,0.439705
max,0.909804,0.870588,0.850980,0.874510,0.866667,0.858824,0.901961,0.937255,0.925490,0.925490,...,0.956863,0.952941,0.960784,0.933333,0.945098,0.941176,0.933333,0.937255,0.960784,0.913725


In [5]:
# Check missing values
print(df.info())

print("Missing vars in columns:\n", df.isna().sum())
print("Number of total missing vars:", df.isna().sum().sum())
print("Number of total missing vars (% of all obs):", (df.isna().sum().sum())/(df.shape[0]*df.shape[1]))


<class 'pandas.core.frame.DataFrame'>
Index: 21105 entries, 0 to 21164
Columns: 1530 entries, Name to PX_3383
dtypes: float32(1528), object(2)
memory usage: 123.5+ MB
None
Missing vars in columns:
 Name       0
Case       0
PX_407     0
PX_408     0
PX_409     0
          ..
PX_3318    0
PX_3319    0
PX_3320    0
PX_3382    0
PX_3383    0
Length: 1530, dtype: int64
Number of total missing vars: 0
Number of total missing vars (% of all obs): 0.0


In [6]:
# 1. Data preprocessing
###

# Create categorical variable from Case
df["Case"] = df.Case.replace({"Normal": 0, "COVID": 1, "Lung_Opacity": 2, "Viral Pneumonia": 3})
df.Case.astype(int)

# Check construction
print(df.Case.value_counts())
print(df.Case.value_counts(normalize = True))

# Split data into target and features
target = df.Case

# Features data: Drop Names and target
data = df.drop(["Name", "Case"], axis = 1)
data.head()
data.shape

# Split data into training and Test sets, save random state
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 123)

Case
0    10191
2     6012
1     3564
3     1338
Name: count, dtype: int64


In [8]:
# 2. Model 1 - Logistic Regression
###
import time
start_time = time.time()

# Instantiate Logistic regression for classification
clf1_name = "Logistic Regression"
clf1 = linear_model.LogisticRegression(solver='lbfgs', C = 1.0, max_iter = 10000)

# Train the model on training data
clf1.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf1.predict(X_test)

# Calc accuracys
clf1_score = clf1.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf1_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model1_time = (time.time() - start_time)/60

In [9]:
# Show Results
###

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

# Score and F1-Score
print("The score is:", clf1_score)
print("The mean F1-Score (unweighted) is:", clf1_f1)

# Show Confusion Matrix
cm1 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm1)

# Show classification report
model1_cr = classification_report(y_test, y_pred)
print(model1_cr)


Model 1: --- 4.8940152088801065 minutes ---
The score is: 0.5979625681118218
The mean F1-Score (unweighted) is: 0.5021242520146714


Predicted Class,0,1,2,3
Realised Class,,,,
0,1653,140,260,50
1,338,104,222,17
2,402,107,631,25
3,66,20,50,136


              precision    recall  f1-score   support

           0       0.67      0.79      0.72      2103
           1       0.28      0.15      0.20       681
           2       0.54      0.54      0.54      1165
           3       0.60      0.50      0.54       272

    accuracy                           0.60      4221
   macro avg       0.52      0.50      0.50      4221
weighted avg       0.57      0.60      0.58      4221



In [10]:
# 3. Model 2 - linear SVM
###
import time
start_time = time.time()

# Instantiate SVM
clf2_name = "linear SVM"
clf2 = svm.SVC(gamma = 0.01, kernel = "poly")

# Train the model on training data
clf2.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf2.predict(X_test)

# Calc accuracy
clf2_score = clf2.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf2_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model2_time = (time.time() - start_time)/60

In [11]:
# Show Results
###

# Modelling Time
print("Model 2: --- %s minutes ---" % model2_time)

# Score and F1-Score
print("The score is:", clf2_score)
print("The mean F1-Score (unweighted) is:", clf2_f1)

# Show Confusion Matrix
cm2 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm2)

# Show classification report
model2_cr = classification_report(y_test, y_pred)
print(model2_cr)

Model 2: --- 7.954864891370137 minutes ---
The score is: 0.6986496090973703
The mean F1-Score (unweighted) is: 0.6050027282424415


Predicted Class,0,1,2,3
Realised Class,,,,
0,1878,64,150,11
1,353,141,177,10
2,307,63,781,14
3,76,11,36,149


              precision    recall  f1-score   support

           0       0.72      0.89      0.80      2103
           1       0.51      0.21      0.29       681
           2       0.68      0.67      0.68      1165
           3       0.81      0.55      0.65       272

    accuracy                           0.70      4221
   macro avg       0.68      0.58      0.61      4221
weighted avg       0.68      0.70      0.67      4221



In [12]:
# 3. Model 3 - KNN
###
from sklearn import neighbors
import time
start_time = time.time()

# Instantiate classifier
clf3_name = "KNN"
clf3 = neighbors.KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

# Train the model on training data
clf3.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf3.predict(X_test)

# Calc accuracys
clf3_score = clf3.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf3_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model3_time = (time.time() - start_time)/60

In [13]:
# Show Results
###

# Modelling Time
print("Model 3: --- %s minutes ---" % model3_time)

# Score and F1-Score
print("The score is:", clf3_score)
print("The mean F1-Score (unweighted) is:", clf3_f1)

# Show Confusion Matrix
cm3 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm3)

# Show classification report
model3_cr = classification_report(y_test, y_pred)
print(model3_cr)

Model 3: --- 0.3723636229832967 minutes ---
The score is: 0.6697465055674011
The mean F1-Score (unweighted) is: 0.5449823778735033


Predicted Class,0,1,2,3
Realised Class,,,,
0,1820,129,146,8
1,308,150,221,2
2,322,70,772,1
3,87,39,61,85


              precision    recall  f1-score   support

           0       0.72      0.87      0.78      2103
           1       0.39      0.22      0.28       681
           2       0.64      0.66      0.65      1165
           3       0.89      0.31      0.46       272

    accuracy                           0.67      4221
   macro avg       0.66      0.52      0.54      4221
weighted avg       0.65      0.67      0.65      4221



In [14]:
# 3. Model 4 - Decision Tree
###
from sklearn.tree import DecisionTreeClassifier
import time
start_time = time.time()

# Instantiate classifier
clf4_name = "Decision Tree"
clf4 = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

# Train the model on training data
clf4.fit(X_train, y_train)

# Make predictions on test set
y_pred = clf4.predict(X_test)

# Calc accuracys
clf4_score = clf4.score(X_test, y_test)

# Calc mean unweighted F1 Score in all classes
clf4_f1 = f1_score(y_test, y_pred, average = "macro")

# Measure time
model4_time = (time.time() - start_time)/60


In [15]:
# Show Results
###

# Modelling Time
print("Model 4: --- %s minutes ---" % model4_time)

# Score and F1-Score
print("The score is:", clf4_score)
print("The mean F1-Score (unweighted) is:", clf4_f1)

# Show Confusion Matrix
cm4 = pd.crosstab(y_test, y_pred, rownames = ['Realised Class'], colnames = ['Predicted Class'])
display(cm4)

# Show classification report
model4_cr = classification_report(y_test, y_pred)
print(model4_cr)
# Ideas: Could show most important Features here. But well, there are 4000 pixels...

Model 4: --- 0.08135678768157958 minutes ---
The score is: 0.546789860222696
The mean F1-Score (unweighted) is: 0.30586557646515566


Predicted Class,0,2
Realised Class,,
0,1354,749
1,194,487
2,211,954
3,186,86


              precision    recall  f1-score   support

           0       0.70      0.64      0.67      2103
           1       0.00      0.00      0.00       681
           2       0.42      0.82      0.55      1165
           3       0.00      0.00      0.00       272

    accuracy                           0.55      4221
   macro avg       0.28      0.37      0.31      4221
weighted avg       0.46      0.55      0.49      4221



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
